In [1]:
from argparse import Namespace

args = Namespace(
    random_seed=2024,
    is_training=1,
    model_id="Electricity_96_192",
    model="CycleNet",
    data="custom",
    root_path="./dataset/",
    data_path="electricity.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",
    seq_len=96,
    label_len=0,
    pred_len=192,
    cycle=168,
    model_type="linear",
    use_revin=0,
    fc_dropout=0.05,
    head_dropout=0.0,
    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=0,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,
    rnn_type="gru",
    dec_way="pmf",
    seg_len=48,
    channel_id=1,
    period_len=24,
    embed_type=0,
    enc_in=321,
    dec_in=7,
    c_out=7,
    d_model=512,
    n_heads=8,
    e_layers=2,
    d_layers=1,
    d_ff=2048,
    moving_avg=25,
    factor=1,
    distil=True,
    dropout=0,
    embed="timeF",
    activation="gelu",
    output_attention=False,
    do_predict=False,
    num_workers=10,
    itr=1,
    train_epochs=30,
    batch_size=64,
    patience=5,
    learning_rate=0.01,
    des="test",
    loss="mse",
    lradj="type3",
    pct_start=0.3,
    use_amp=False,
    use_gpu=False,
    gpu=0,
    use_multi_gpu=False,
    devices="0,1",
    test_flop=False,
)

In [2]:
import torch

y = torch.load("CycleNet-out", weights_only=False)
x = torch.load("CycleNet-in-x", weights_only=False)
cycle_index = torch.load("CycleNet-in-cycle_index", weights_only=False)

In [3]:
import torch
import torch.nn as nn


class RecurrentCycle(torch.nn.Module):
    # Thanks for the contribution of wayhoww.
    # The new implementation uses index arithmetic with modulo to directly gather cyclic data in a single operation,
    # while the original implementation manually rolls and repeats the data through looping.
    # It achieves a significant speed improvement (2x ~ 3x acceleration).
    # See https://github.com/ACAT-SCUT/CycleNet/pull/4 for more details.
    def __init__(self, cycle_len, channel_size):
        super(RecurrentCycle, self).__init__()
        self.cycle_len = cycle_len
        self.channel_size = channel_size
        self.data = torch.nn.Parameter(
            torch.zeros(cycle_len, channel_size), requires_grad=True
        )

    def forward(self, index, length):
        # index B
        # index.view(-1, 1) B,1
        
        gather_index = (
            index.view(-1, 1) + torch.arange(length, device=index.device).view(1, -1)
        ) % self.cycle_len
        return self.data[gather_index]


class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()

        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.enc_in = configs.enc_in
        self.cycle_len = configs.cycle
        self.model_type = configs.model_type
        self.d_model = configs.d_model
        self.use_revin = configs.use_revin

        self.cycleQueue = RecurrentCycle(
            cycle_len=self.cycle_len, channel_size=self.enc_in
        )
        self.model = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x, cycle_index):
        # x: (batch_size, seq_len, enc_in), cycle_index: (batch_size,)

        # instance norm
        if self.use_revin:
            seq_mean = torch.mean(x, dim=1, keepdim=True)
            seq_var = torch.var(x, dim=1, keepdim=True) + 1e-5
            x = (x - seq_mean) / torch.sqrt(seq_var)

        # remove the cycle of the input data
        x = x - self.cycleQueue(cycle_index, self.seq_len)

        print(self.cycleQueue(cycle_index, self.seq_len).shape)

        # forecasting with channel independence (parameters-sharing)
        y = self.model(x.permute(0, 2, 1)).permute(0, 2, 1)

        # add back the cycle of the output data
        y = y + self.cycleQueue(
            (cycle_index + self.seq_len) % self.cycle_len, self.pred_len
        )

        # instance denorm
        if self.use_revin:
            y = y * torch.sqrt(seq_var) + seq_mean

        return y


model = Model(args)

_ = model(x, cycle_index)

torch.Size([64, 96, 321])


In [4]:
model.cycleQueue.data.shape

torch.Size([168, 321])

In [5]:
model.cycleQueue.cycle_len

168

In [14]:
cycle_index,cycle_index.shape

(tensor([  7,  81,   4, 151, 151, 149,  22,  60, 126,  11,   1, 150, 126,  85,
         108,  68, 130, 161, 134,  74,   7, 128, 148,  26,   4,  69, 146,   2,
          35,  55,  75, 150,  63,  67,  41,  36,  32, 134, 155, 143,   7,  98,
          86,  17, 124, 115, 127,  94,  38,  71,  66,  55,  65,  88, 145, 101,
         102,  24, 132,  87,  57, 125,  15, 166], dtype=torch.int32),
 torch.Size([64]))

In [8]:
from data_provider.data_factory import data_provider

dataset, dataloader = data_provider(args, "train")

train 18125


In [13]:
dataset[4][-1]

tensor(100, dtype=torch.int32)